# 30-Day Hospital Readmission Risk: Modeling and Evaluation
### SWYNEX Technologies — Task 3: Machine Learning Modeling & Clinical Evaluation
**Author:** Abhijeet Arun Gore  
**Role:** Data Science Intern at SWYNEX Technologies  
**Dataset:** Diabetes 130-US Hospitals (1999–2008), Cleaned Inpatient Dataset from Task 1  
**Repository:** [SWYNEX-Data-Preparation](https://github.com/AbhijeetArunGore/SWYNEX-Data-Preparation)  

---

## Section 1: Introduction

### The Clinical Prediction Problem
Hospital readmission within 30 days of discharge is an essential quality and cost metric in modern healthcare systems. Under the **Hospital Readmissions Reduction Program (HRRP)** established by the Centers for Medicare & Medicaid Services (CMS), hospitals face financial penalties for elevated 30-day readmission rates for chronic conditions, including diabetes and heart failure.

In this project, our goal is to build a transparent, baseline machine learning classification model that predicts whether an inpatient diabetic encounter is followed by an unplanned **readmission within 30 days (`<30`)**.

### Dataset and Scope
We build directly upon the work completed in previous tasks:
- **Task 1 (Data Preparation)**: Cleaned 101,763 hospital encounters, standardized implicit missingness (`?`), handled zero-variance columns, and mapped relational diagnosis lookups (`IDS_mapping.csv`).
- **Task 2 (Exploratory Data Analysis)**: Documented an ~8:1 target class imbalance, identified prior inpatient visits as the primary risk driver, and discovered critical data leakage risks (including in-hospital deceased records and repeat patient clustering).

### Guiding Principles for Task 3
1. **Honest Student Modeling**: We avoid unnecessarily complex black-box ensembles or deep learning. We implement a solid, understandable **Logistic Regression baseline with balanced class weights** and benchmark it against a majority-class **DummyClassifier**.
2. **Strict Anti-Leakage Governance**: We implement a patient-grouped holdout split (`GroupShuffleSplit`) to prevent encounters from the same patient from leaking across partitions.
3. **Clinical Metric Transparency**: In imbalanced medical datasets, raw accuracy is deceptive. We evaluate precision, recall, F1-score, ROC-AUC, and Precision-Recall AUC (PR-AUC).
4. **Educational Scope**: This model is a learning project. It is **not clinically validated** or ready for real-time bedside deployment.


---
## Section 2: Load and Inspect the Data

We load our cleaned dataset (`data/processed/diabetic_data_clean.csv`) exported in Task 1. We inspect the dataset shape, confirm the presence of the `readmitted` target column, verify that there are zero unhandled null values or duplicate rows, and inspect the target class balance.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure access to modular utility functions
sys.path.insert(0, os.path.abspath('..'))
from src.modeling import (
    map_binary_target,
    prepare_feature_matrix,
    create_patient_grouped_split,
    build_logistic_regression_pipeline,
    evaluate_model
)

# Configure styling aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Segoe UI'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
pd.set_option('display.max_columns', 60)

# Load data with robust relative path handling
data_paths = [
    os.path.join('data', 'processed', 'diabetic_data_clean.csv'),
    os.path.join('..', 'data', 'processed', 'diabetic_data_clean.csv')
]
csv_path = next(p for p in data_paths if os.path.exists(p))

df = pd.read_csv(csv_path)
print(f"Dataset successfully loaded from: {csv_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Total Missing Values: {df.isnull().sum().sum()}")
print(f"Exact Duplicate Rows: {df.duplicated().sum()}")
print(f"Unique Patients: {df['patient_nbr'].nunique():,}")

# Confirm raw target values
print("\n=== RAW TARGET DISTRIBUTION ('readmitted') ===")
raw_target_counts = df['readmitted'].value_counts()
raw_target_pcts = df['readmitted'].value_counts(normalize=True) * 100
for k, v in raw_target_counts.items():
    print(f"  {k:5s}: {v:6,d} ({raw_target_pcts[k]:.2f}%)")


### Binary Target Definition:
- **Positive Class ($y = 1$)**: `readmitted == '<30'` — Readmission occurred within 30 days of discharge.
- **Negative Class ($y = 0$)**: `readmitted in {'NO', '>30'}` — No readmission was recorded, or readmission occurred after 30 days.

*Note on 'NO'*: An outcome of `NO` indicates that no readmission was observed in this hospital network. It does not prove that the patient was never hospitalized elsewhere.


In [ ]:
# Map binary target
df['target'] = (df['readmitted'] == '<30').astype(int)
pos_count = int(df['target'].sum())
neg_count = int(len(df) - pos_count)

print("=== BINARY TARGET CLASSIFICATION BREAKDOWN ===")
print(f"Negative Class (0 - Not Readmitted <30 Days): {neg_count:,} ({neg_count/len(df)*100:.2f}%)")
print(f"Positive Class (1 - Readmitted Within 30 Days): {pos_count:,} ({pos_count/len(df)*100:.2f}%)")
print(f"Imbalance Ratio: Approximately {neg_count / pos_count:.1f} to 1")


---
## Section 3: Prepare the Features

### Feature Segregation and Leakage Prevention
To prevent data leakage and ensure realistic model inputs, we strictly exclude the following columns:
1. **Identifiers**: `encounter_id` and `patient_nbr`. `patient_nbr` is preserved strictly as the grouping key for train-test splitting.
2. **Raw Outcome**: `readmitted` and temporary `target`.
3. **Confirmed Target Leakage**: `discharge_disposition_id` and `discharge_disposition_desc`.
   - In Task 2, we showed that encounters resulting in inpatient death (`Expired`, $N=1,652$) all had an exact 0.0% readmission rate.
   - Including `discharge_disposition` allows models to predict "no readmission" from the fact that a patient died.
   - Furthermore, discharge disposition is determined at the very end of hospitalization; excluding it aligns with the operational goal of identifying risk earlier during the stay.
4. **Redundant High-Cardinality Codes**: Raw `diag_1`, `diag_2`, `diag_3` (900+ codes) are captured by standard categories (`diag_1_category`, etc.), and integer ID codes (`admission_type_id`, `admission_source_id`) are captured by clean text descriptions.

All transformations (`StandardScaler` for continuous features, `OneHotEncoder(handle_unknown='ignore')` for categorical features) are assembled in a scikit-learn `ColumnTransformer` to be fitted **strictly on training data only**.


In [ ]:
# Separate features, target, and grouping key
X, y, groups, num_cols, cat_cols = prepare_feature_matrix(df, target_col='readmitted', exclude_leakage=True)

print(f"Input Feature Matrix (X): {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Numerical Features ({len(num_cols)}): {num_cols}")
print(f"Categorical Features ({len(cat_cols)}): {len(cat_cols)} categorical attributes")
print(f"Sample Categorical: {cat_cols[:6]}...")


---
## Section 4: Train-Test Split

### Patient-Grouped Splitting (Preventing Intra-Patient Leakage)
In our dataset, **29.72%** of encounters represent repeat admissions by recurring patients. If we used standard random splitting, encounters from the same patient would end up in both training and test sets. The model could simply memorize patient identities rather than generalizable clinical patterns.

We use scikit-learn's **`GroupShuffleSplit`** with `test_size=0.20` and `random_state=42`, grouping strictly on `patient_nbr`.


In [ ]:
# Create patient-grouped holdout split
X_train, X_test, y_train, y_test = create_patient_grouped_split(
    X, y, groups=groups, test_size=0.20, random_state=42
)

# Verify zero patient overlap
train_patients = set(groups.iloc[X_train.index])
test_patients = set(groups.iloc[X_test.index])
overlap = train_patients.intersection(test_patients)

print("=== PATIENT-GROUPED SPLIT VERIFICATION ===")
print(f"Training Encounters: {len(X_train):,} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Train Positive Rate: {y_train.mean()*100:.2f}% ({y_train.sum():,} positive)")
print(f"Held-Out Test Encounters: {len(X_test):,} ({len(X_test)/len(X)*100:.1f}%)")
print(f"  Test Positive Rate: {y_test.mean()*100:.2f}% ({y_test.sum():,} positive)")
print(f"Patient Overlap Between Train and Test: {len(overlap)} (Zero Leakage Confirmed)")


---
## Section 5: Establish a Baseline (DummyClassifier)

Before training any complex model, a good data scientist always establishes a naive baseline. Here we use scikit-learn's **`DummyClassifier`** configured with `strategy='most_frequent'`, which always predicts the dominant class (0 — Not Readmitted within 30 Days).


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Fit majority-class dummy baseline on training data
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

# Evaluate on held-out test data
y_pred_dummy = dummy.predict(X_test)

dummy_acc = accuracy_score(y_test, y_pred_dummy)
dummy_prec = precision_score(y_test, y_pred_dummy, zero_division=0)
dummy_rec = recall_score(y_test, y_pred_dummy, zero_division=0)
dummy_f1 = f1_score(y_test, y_pred_dummy, zero_division=0)

print("=== BASELINE DUMMY CLASSIFIER (MOST FREQUENT) METRICS ===")
print(f"Accuracy:  {dummy_acc:.4f} ({dummy_acc*100:.2f}%)")
print(f"Precision: {dummy_prec:.4f}")
print(f"Recall:    {dummy_rec:.4f} (0.0% of positive cases detected)")
print(f"F1-Score:  {dummy_f1:.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dummy))


### Why Accuracy is Deceptive on Imbalanced Datasets
Notice that the DummyClassifier achieves an accuracy of **89.08%**! To an untrained observer, an 89% accuracy might look like a working model.

However, examining the confusion matrix reveals that the model predicted `0` for all 20,093 test encounters:
- **True Negatives**: 17,898 (all negative cases)
- **True Positives**: **0**
- **False Negatives**: **2,195** (missed 100% of the actual 30-day readmissions)

In a hospital setting, this model is completely useless. It fails to identify a single high-risk patient who needs post-discharge transitional care. This illustrates why **Recall**, **Precision**, and **PR-AUC** are the essential evaluation metrics for imbalanced healthcare datasets.


---
## Section 6: Train a Simple Machine Learning Model (Logistic Regression)

### Algorithm Choice
We select **Logistic Regression** as our primary machine learning model. Logistic Regression is:
- Fast and computationally efficient.
- Interpretable, allowing inspection of standardized feature log-odds coefficients.
- Transparent for pair programming and peer review.

### Preprocessing and Class Imbalance Handling
1. **Numerical Features**: Scaled using `StandardScaler` (zero mean, unit variance) so features with larger ranges (like `num_lab_procedures` up to 132) do not dominate features with small counts (like `number_inpatient`).
2. **Categorical Features**: Encoded using `OneHotEncoder(handle_unknown='ignore')` to handle unseen categories at test time gracefully.
3. **Class Weighting**: We set `class_weight='balanced'`. This automatically calculates class weights inversely proportional to class frequencies:
   $$\text{weight}(c) = \frac{N}{2 \times N_c}$$
   For our positive class, this assigns approximately **~8x higher penalty** to False Negatives, forcing the model to pay attention to readmitted patients.


In [ ]:
from sklearn.linear_model import LogisticRegression

# Construct the full scikit-learn pipeline
lr_pipeline = build_logistic_regression_pipeline(
    numerical_cols=num_cols,
    categorical_cols=cat_cols,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

print("Pipeline Architecture:")
print(lr_pipeline)

# Fit strictly on the training partition
print("\nFitting Balanced Logistic Regression on X_train (81,670 encounters)...")
lr_pipeline.fit(X_train, y_train)
print("Model training complete.")


---
## Section 7: Evaluate the Model

We evaluate both the Dummy Baseline and the Logistic Regression model on the exact same **held-out test set of 20,093 encounters**.

We calculate:
- **Accuracy**: Overall fraction of correct predictions.
- **Precision**: When the model flags an encounter as high-risk, how often is it right?
- **Recall (Sensitivity)**: What percentage of actual 30-day readmissions did the model capture?
- **F1-Score**: Harmonic mean of Precision and Recall.
- **ROC-AUC**: Area under the Receiver Operating Characteristic curve.
- **PR-AUC (Average Precision)**: Area under the Precision-Recall curve, specifically calibrated for imbalanced data.


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

# Generate predictions and probabilities for test partition
y_pred_lr = lr_pipeline.predict(X_test)
y_prob_lr = lr_pipeline.predict_proba(X_test)[:, 1]
y_prob_dummy = dummy.predict_proba(X_test)[:, 1]

# Evaluate both models using modular utility
dummy_metrics = evaluate_model(dummy, X_test, y_test)
lr_metrics = evaluate_model(lr_pipeline, X_test, y_test)

metrics_comparison = pd.DataFrame([
    {
        'Evaluation Metric': 'Accuracy',
        'Dummy Baseline': f"{dummy_metrics['accuracy']:.4f}",
        'Balanced Logistic Regression': f"{lr_metrics['accuracy']:.4f}",
        'Clinical Significance': 'Baseline has high accuracy by guessing 0; model trades overall accuracy for sensitivity.'
    },
    {
        'Evaluation Metric': 'Precision',
        'Dummy Baseline': f"{dummy_metrics['precision']:.4f}",
        'Balanced Logistic Regression': f"{lr_metrics['precision']:.4f}",
        'Clinical Significance': '16.48% of flagged patients readmit within 30 days (~1 in 6 alarms is true).'
    },
    {
        'Evaluation Metric': 'Recall (Sensitivity)',
        'Dummy Baseline': f"{dummy_metrics['recall']:.4f}",
        'Balanced Logistic Regression': f"{lr_metrics['recall']:.4f}",
        'Clinical Significance': 'Model catches 1,147 of 2,195 (52.26%) true 30-day readmissions.'
    },
    {
        'Evaluation Metric': 'F1-Score',
        'Dummy Baseline': f"{dummy_metrics['f1_score']:.4f}",
        'Balanced Logistic Regression': f"{lr_metrics['f1_score']:.4f}",
        'Clinical Significance': 'Harmonic balance between precision and recall on the positive class.'
    },
    {
        'Evaluation Metric': 'ROC-AUC',
        'Dummy Baseline': f"{dummy_metrics['roc_auc']:.4f}",
        'Balanced Logistic Regression': f"{lr_metrics['roc_auc']:.4f}",
        'Clinical Significance': 'Overall discriminatory capability across thresholds (0.50 = random chance).'
    },
    {
        'Evaluation Metric': 'PR-AUC (Average Precision)',
        'Dummy Baseline': f"{dummy_metrics['pr_auc']:.4f}",
        'Balanced Logistic Regression': f"{lr_metrics['pr_auc']:.4f}",
        'Clinical Significance': '+72.8% relative improvement over test positive base rate (0.1092).'
    }
])

print("=== SIDE-BY-SIDE MODEL EVALUATION ON HELD-OUT TEST COHORT (N=20,093) ===")
metrics_comparison


In [ ]:
print("=== DETAILED CLASSIFICATION REPORT: LOGISTIC REGRESSION ===")
print(classification_report(y_test, y_pred_lr, digits=4, zero_division=0))


---
## Section 8: Visualize the Results

We create four key visualizations to communicate model behavior:
1. **Confusion Matrix Comparison**: Visualizes True Positives, False Positives, False Negatives, and True Negatives.
2. **Metrics Bar Chart**: Side-by-side comparison across all 6 performance metrics.
3. **ROC and Precision-Recall Curves**: Illustrates trade-offs across all continuous classification thresholds.
4. **Top Feature Coefficients**: Displays the strongest positive and negative predictors of 30-day readmission.


In [ ]:
# 1. Confusion Matrix Comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

sns.heatmap(np.array(dummy_metrics['confusion_matrix']), annot=True, fmt=',d', cmap='Blues', cbar=False,
            xticklabels=['Pred 0 (No)', 'Pred 1 (<30)'], yticklabels=['True 0 (No)', 'True 1 (<30)'],
            ax=axes[0], annot_kws={'size': 11, 'weight': 'bold'})
axes[0].set_title('A: Baseline DummyClassifier
(Recall = 0.0%, Misses All 2,195 Cases)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=10)
axes[0].set_ylabel('True Label', fontsize=10)

sns.heatmap(np.array(lr_metrics['confusion_matrix']), annot=True, fmt=',d', cmap='Blues', cbar=False,
            xticklabels=['Pred 0 (No)', 'Pred 1 (<30)'], yticklabels=['True 0 (No)', 'True 1 (<30)'],
            ax=axes[1], annot_kws={'size': 11, 'weight': 'bold'})
axes[1].set_title(f"B: Balanced Logistic Regression\n(Recall = 52.26%, Catches 1,147 Cases)", fontsize=11, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=10)
axes[1].set_ylabel('True Label', fontsize=10)

plt.suptitle('Held-Out Test Set Confusion Matrix Comparison (N=20,093)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 2. Metric Comparison Bar Chart
metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC']
dummy_scores = [dummy_metrics['accuracy'], dummy_metrics['precision'], dummy_metrics['recall'], dummy_metrics['f1_score'], dummy_metrics['roc_auc'], dummy_metrics['pr_auc']]
lr_scores = [lr_metrics['accuracy'], lr_metrics['precision'], lr_metrics['recall'], lr_metrics['f1_score'], lr_metrics['roc_auc'], lr_metrics['pr_auc']]

x = np.arange(len(metric_labels))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width/2, dummy_scores, width, label='Dummy Baseline (Most Frequent)', color='#7f8c8d', edgecolor='black', linewidth=0.5)
plt.bar(x + width/2, lr_scores, width, label='Balanced Logistic Regression', color='#2b5c8f', edgecolor='black', linewidth=0.5)

plt.title('Performance Comparison: Baseline vs. Balanced Logistic Regression', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Metric', fontsize=11, fontweight='semibold')
plt.ylabel('Score (0.0 to 1.0)', fontsize=11, fontweight='semibold')
plt.xticks(x, metric_labels, fontsize=10, fontweight='semibold')
plt.ylim(0, 1.05)
plt.legend(frameon=True, loc='upper right')

for i, (d, l) in enumerate(zip(dummy_scores, lr_scores)):
    plt.text(i - width/2, d + 0.02, f'{d:.3f}', ha='center', va='bottom', fontsize=8.5, color='#555555')
    plt.text(i + width/2, l + 0.02, f'{l:.3f}', ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#1a3a5a')

plt.tight_layout()
plt.show()


In [ ]:
# 3. ROC and Precision-Recall Curves
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob_lr)
axes[0].plot(fpr, tpr, color='#2b5c8f', linewidth=2.5, label=f'Logistic Regression (AUC = {lr_metrics["roc_auc"]:.4f})')
axes[0].plot([0, 1], [0, 1], color='#7f8c8d', linestyle='--', linewidth=1.5, label='Random Chance (AUC = 0.5000)')
axes[0].set_title('A: Receiver Operating Characteristic (ROC) Curve', fontsize=11, fontweight='bold')
axes[0].set_xlabel('False Positive Rate', fontsize=10)
axes[0].set_ylabel('True Positive Rate (Recall)', fontsize=10)
axes[0].legend(loc='lower right', frameon=True)

# PR Curve
prec_vals, rec_vals, _ = precision_recall_curve(y_test, y_prob_lr)
base_rate = float(y_test.mean())
axes[1].plot(rec_vals, prec_vals, color='#c0392b', linewidth=2.5, label=f'Logistic Regression (PR-AUC = {lr_metrics["pr_auc"]:.4f})')
axes[1].axhline(y=base_rate, color='#7f8c8d', linestyle='--', linewidth=1.5, label=f'Test Positive Base Rate ({base_rate:.4f})')
axes[1].set_title('B: Precision-Recall (PR) Curve', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Recall (Sensitivity)', fontsize=10)
axes[1].set_ylabel('Precision (Positive Predictive Value)', fontsize=10)
axes[1].legend(loc='upper right', frameon=True)

plt.suptitle('Discrimination Curves on Held-Out Test Cohort', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# 4. Top Feature Coefficients from Logistic Regression
cat_encoder = lr_pipeline.named_steps['prep'].named_transformers_['cat']
cat_feature_names = cat_encoder.get_feature_names_out(cat_cols)
all_feature_names = num_cols + list(cat_feature_names)

coefs = lr_pipeline.named_steps['clf'].coef_[0]
coef_df = pd.DataFrame({'feature': all_feature_names, 'coefficient': coefs})

top_pos = coef_df.sort_values(by='coefficient', ascending=False).head(8)
top_neg = coef_df.sort_values(by='coefficient', ascending=True).head(8)
plot_coef_df = pd.concat([top_neg, top_pos]).sort_values(by='coefficient', ascending=True)

plot_coef_df['display'] = plot_coef_df['feature'].apply(
    lambda x: x.replace('medical_specialty_', 'Spec: ').replace('admission_type_desc_', 'Adm: ').replace('payer_code_', 'Payer: ')
)

plt.figure(figsize=(10, 6))
bar_colors = ['#27ae60' if c < 0 else '#c0392b' for c in plot_coef_df['coefficient']]
plt.barh(plot_coef_df['display'], plot_coef_df['coefficient'], color=bar_colors, edgecolor='black', linewidth=0.5)

plt.title('Top Positive and Negative Logistic Regression Coefficients
(Predicting 30-Day Early Hospital Readmission)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Standardized Log-Odds Coefficient', fontsize=11, fontweight='semibold')
plt.ylabel('Clinical Feature', fontsize=11, fontweight='semibold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

for bar in plt.gca().patches:
    w = bar.get_width()
    ha = 'left' if w >= 0 else 'right'
    offset = 0.03 if w >= 0 else -0.03
    plt.text(w + offset, bar.get_y() + bar.get_height()/2.0, f'{w:.2f}', va='center', ha=ha, fontsize=8.5, fontweight='bold')

plt.tight_layout()
plt.show()
